# Merge Lora

In [31]:
!pip -q uninstall -y torch torchvision torchaudio
!pip -q uninstall -y transformers

# Install latest stable PyTorch CUDA build (pick one from pytorch.org; example uses CUDA 12.6 channel)
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

# Then install a stable Transformers/PEFT stack
!pip -q install "transformers>=4.51.0" "peft>=0.10.0" "accelerate>=0.33.0" "safetensors"

You can safely remove it manually.
You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires transformers, which is not installed.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, which is not installed.
trl 0.28.0 requires transformers>=4.56.2, which is not installed.


In [1]:
!pip -q uninstall -y huggingface_hub transformers peft trl accelerate
!pip -q install -U "huggingface_hub>=0.34,<1.0" "transformers>=4.56" "peft" "trl" "accelerate"

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE = "google/gemma-3-270m-it"
ADAPTER_DIR = "gemma270_lora_out"
OUT_DIR = "lambda_build/merged"
HF_TOKEN = "" # need huggingface token that can access google gemma hf model

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=False, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    trust_remote_code=False,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN
)

model = PeftModel.from_pretrained(model, ADAPTER_DIR)
# merge LoRA into base weights
model = model.merge_and_unload()

model.save_pretrained(OUT_DIR, safe_serialization=True)
tokenizer.save_pretrained(OUT_DIR)

print("Merged model saved to:", OUT_DIR)

Merged model saved to: lambda_build/merged


# Convert merged HF model → GGUF (F16)

In [3]:
!python -m pip install -r lambda_build/llama.cpp/requirements.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
  Using cached https://download.pytorch.org/whl/cpu/torch-2.6.0%2Bcpu-cp312-cp312-win_amd64.whl.metadata (28 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
Using cached transformers-4.57.6-py3-none-any.whl (12.0 MB)
Using cached https://download.pytorch.org/whl/cpu/torch-2.6.0%2Bcpu-cp312-cp312-win_amd64.whl (206.5 MB)
Using cached sympy-1.13.1-py3-none-any.whl (6.2 MB)
Using cached hugg

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.5.0 requires fsspec[http]<=2025.10.0,>=2023.1.0, but you have fsspec 2025.12.0 which is incompatible.
torchaudio 2.10.0+cu126 requires torch==2.10.0, but you have torch 2.6.0+cpu which is incompatible.
torchvision 0.25.0+cu126 requires torch==2.10.0, but you have torch 2.6.0+cpu which is incompatible.


In [7]:
import os
from transformers import AutoTokenizer

BASE = "google/gemma-3-270m-it"
MERGED_DIR = r"lambda_build/merged"

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
tok.save_pretrained(MERGED_DIR)

print("Tokenizer files in merged dir:")
print("\n".join(sorted(os.listdir(MERGED_DIR))))

Tokenizer files in merged dir:
added_tokens.json
chat_template.jinja
config.json
generation_config.json
model.safetensors
special_tokens_map.json
tokenizer.json
tokenizer.model
tokenizer_config.json


In [9]:
!python lambda_build/llama.cpp/convert_hf_to_gguf.py \
    lambda_build/merged \
    --outfile lambda_build/gguf/gemma3_merged_f16.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: merged
INFO:hf-to-gguf:Model architecture: Gemma3ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,                 torch.float16 --> F16, shape = {640, 262144}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.float16 --> F32, shape = {640}
INFO:hf-to-gguf:blk.0.ffn_down.weight,             torch.float16 --> F16, shape = {2048, 640}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,             torch.float16 --> F16, shape = {640, 2048}
INFO:hf-to-gguf:blk.0.ffn_up.weight,               torch.float16 --> F16, shape = {640, 2048}
INFO:hf-to-gguf:blk.0.post_attention_norm.weight,  torch.float16 --> F32, shape = {640}
INFO:hf-to-gguf:blk.0.post_ffw_norm.weight,        torch.float16 --> F32, shape = {640}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,             torch.float16 --> F32, shape = {640}
INFO:hf-to-g

In [ ]:
# Run this line in Ubuntu command prompt, at the folder "lambda_build/llama.cpp/build/bin/"
# ./llama-quantize ../../../gguf/gemma3_merged_f16.gguf ../../../gguf/gemma3_merged_q4_k_m.gguf Q4_K_M

# Further quantization

In [ ]:
# ./llama-quantize ../../../gguf/gemma3_merged_f16.gguf ../../../gguf/gemma3_merged_q3_k_m.gguf Q3_K_M
# ./llama-quantize ../../../gguf/gemma3_merged_f16.gguf ../../../gguf/gemma3_merged_q2.gguf Q2_K